## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

SEED = 42
DELAY_THRESHOLD_MIN = 15            # delayed = BLOCK_ON more than 15 minutes after STA
EXCLUDE_TEST_REMARK_ROWS = False    # your decision after Cell 6
DATA_PATH = Path("../data/aibd_2025_regular_arrivals.csv")

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)
print("Data file found:", DATA_PATH.exists())

pandas 2.3.3 | numpy 2.3.5 | scikit-learn 1.7.2
Data file found: True


## 2. Data Preparation

This section loads the AIBD regular arrivals dataset and performs an initial validation of its structure. Before any modelling work, it is important to confirm that key fields such as scheduled arrival time (STA) and block-on time are complete and that each flight event is uniquely identified.

The checks performed here ensure that the dataset is internally consistent and suitable for delay analysis. Any violation of these assumptions would need to be investigated before continuing with feature engineering or model development.

In [2]:
raw = pd.read_csv(
    DATA_PATH,
    parse_dates=["sta_datetime", "ata_datetime", "block_on_datetime"],
    low_memory=False,
)

assert len(raw) == 10_863, f"Unexpected row count: {len(raw)}"
assert raw["aibd_event_id"].is_unique, "Event identifiers are not unique"
assert raw["is_regular_arrival"].all() and not raw["is_cancelled"].any()
assert raw[["sta_datetime", "block_on_datetime"]].notna().all().all(), "Missing STA or BLOCK_ON"

print("Rows:", len(raw), "| Columns:", raw.shape[1])
print("Scheduled arrivals from", raw["sta_datetime"].min(), "to", raw["sta_datetime"].max())

Rows: 10863 | Columns: 60
Scheduled arrivals from 2025-01-01 00:05:00 to 2025-12-31 23:25:00


## 3. Column Inventory

Before building a predictive model, each variable must be assessed according to when the information becomes available. The objective is to ensure that only information known before the aircraft arrives can be used as a predictor.

This inventory separates identifiers, candidate predictors, target-related variables, and operational fields that could introduce data leakage. Establishing these roles at an early stage helps maintain a realistic prediction framework.

In [3]:
ROLES = {
    "identifier": ["aibd_event_id", "source_excel_row"],
    "predictor candidate (known before arrival)": [
        "airline", "flight_number", "aircraft_type", "provenance", "sta_datetime"],
    "target source (never a feature)": ["block_on_datetime", "sta_to_block_on_min"],
    "excluded: known only at or after arrival": [
        "ata_raw", "ata_datetime", "block_on_raw", "sta_to_ata_min", "ata_to_block_on_min",
        "arrival_delay_raw", "arrival_delay_code",
        "pax_in", "terminating_passengers", "bb_in_raw", "bb_in_effective",
        "bb_in_was_missing", "bb_in_zero_imputation_supported",
        "trans_raw", "trans_effective_for_aggregation", "trans_was_missing",
        "stand", "belt", "gate", "remarks"],
    "excluded: redundant or bookkeeping": [
        "operational_date", "sta_raw", "is_simulation_replay_eligible", "aircraft_registration"],
}
assigned = {col for cols in ROLES.values() for col in cols}
constant = [c for c in raw.columns if c not in assigned and raw[c].nunique(dropna=True) <= 1]
ROLES["excluded: empty or constant"] = constant

role_of = {col: role for role, cols in ROLES.items() for col in cols}
unassigned = [c for c in raw.columns if c not in role_of]
assert not unassigned, f"Columns without a role: {unassigned}"
assert all(c in raw.columns for c in role_of), "A listed column is not in the file"

inventory = pd.Series(role_of).value_counts().reindex(ROLES.keys()).rename("columns")
print(inventory.to_string())
print("\nTotal:", inventory.sum(), "of", raw.shape[1], "columns")

identifier                                     2
predictor candidate (known before arrival)     5
target source (never a feature)                2
excluded: known only at or after arrival      20
excluded: redundant or bookkeeping             4
excluded: empty or constant                   27

Total: 60 of 60 columns


Stand assignment was excluded despite its potential operational relevance. The dataset does not document whether the recorded stand is the planned allocation or the stand actually used, and stand allocations can be changed in response to delays, congestion, and resource conflicts. The recorded value may therefore reflect decisions taken after a delay became evident, which would create a risk of target leakage. Because this cannot be verified from the data, the stand was treated as unavailable at prediction time.

In [4]:
reg_counts = raw["aircraft_registration"].value_counts()
print("Registrations:", len(reg_counts),
      "| seen fewer than 5 times:", int((reg_counts < 5).sum()),
      "| median flights per registration:", int(reg_counts.median()))

Registrations: 609 | seen fewer than 5 times: 218 | median flights per registration: 7


Aircraft registration was excluded from modelling. Although it is known before arrival, the variable has 609 distinct values and functions largely as an aircraft identifier rather than an operational characteristic. Including such a high-cardinality identifier would increase the risk of overfitting and reduce the model's ability to generalise to aircraft with limited historical observations.

## 4. Quality profile of the predictor candidates

In [5]:
CATEGORICAL = ["airline", "flight_number", "aircraft_type", "provenance"]

def profile(frame):
    return pd.DataFrame({
        "missing": frame[CATEGORICAL].isna().sum(),
        "distinct": frame[CATEGORICAL].nunique(),
        "top_value_share": [frame[c].value_counts(normalize=True).iloc[0] for c in CATEGORICAL],
        "values_seen_under_5_times": [(frame[c].value_counts() < 5).sum() for c in CATEGORICAL],
    }).round(3)

profile_before = profile(raw)
print(profile_before.to_string())

               missing  distinct  top_value_share  values_seen_under_5_times
airline              0        34            0.226                          0
flight_number        1       203            0.034                         83
aircraft_type        6        40            0.260                          5
provenance           0        72            0.090                         13


## 5. Cleaning and the working table

In [6]:
def normalize_text(series):
    """Trim, collapse inner whitespace and uppercase, so 'Air Senegal ' equals 'AIR SENEGAL'."""
    return (series.astype("string")
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True)
                  .str.upper())

data = raw[["aibd_event_id", "sta_datetime", "block_on_datetime", "remarks"] + CATEGORICAL].copy()
for col in CATEGORICAL + ["remarks"]:
    data[col] = normalize_text(data[col])

data[CATEGORICAL] = data[CATEGORICAL].fillna("UNKNOWN")
data = data.rename(columns={"provenance": "origin"})
CATEGORICAL = ["airline", "flight_number", "aircraft_type", "origin"]

# Delay in minutes, recomputed from the timestamps and checked against the stored column
data["delay_min"] = (data["block_on_datetime"] - data["sta_datetime"]).dt.total_seconds() / 60
assert np.allclose(data["delay_min"], raw["sta_to_block_on_min"]), "Stored delay differs from recomputed delay"

data = data.sort_values("sta_datetime", kind="stable").reset_index(drop=True)

change = pd.DataFrame({
    "distinct_before": profile_before["distinct"].values,
    "distinct_after": [data[c].nunique() for c in CATEGORICAL],
    "filled_as_UNKNOWN": [(data[c] == "UNKNOWN").sum() for c in CATEGORICAL],
}, index=CATEGORICAL)
print(change.to_string())
print("\nWorking table:", data.shape[0], "rows x", data.shape[1], "columns")
print(data.dtypes.astype(str).to_string())

               distinct_before  distinct_after  filled_as_UNKNOWN
airline                     34              33                  0
flight_number              203             199                  1
aircraft_type               40              41                  6
origin                      72              70                  0

Working table: 10863 rows x 9 columns
aibd_event_id                object
sta_datetime         datetime64[ns]
block_on_datetime    datetime64[ns]
remarks                      string
airline                      string
flight_number                string
aircraft_type                string
origin                       string
delay_min                   float64


## 6. Row Audit

Not all records necessarily represent standard operational situations. This audit identifies unusual observations such as test records, duplicated schedules, reprogrammed flights, and extreme early arrivals.

The purpose is not immediately to remove these records but to understand their potential influence on the delay distribution. Any exclusion should be justified using operational reasoning rather than statistical convenience alone.

In [7]:
flags = {
    "remark = TEST": data["remarks"].eq("TEST").fillna(False),
    "remark starts with REPROGRAMM": data["remarks"].str.startswith("REPROGRAMM").fillna(False),
    "same flight number and STA": data.duplicated(["flight_number", "sta_datetime"], keep=False),
    "more than 120 min early": data["delay_min"] < -120,
}
audit = pd.DataFrame({
    "rows": {k: int(v.sum()) for k, v in flags.items()},
    "median_delay_min": {k: data.loc[v, "delay_min"].median() for k, v in flags.items()},
})
print(audit.to_string())
print("All flights, median delay:", data["delay_min"].median())

rows_before = len(data)
if EXCLUDE_TEST_REMARK_ROWS:
    data = data.loc[~flags["remark = TEST"].to_numpy()].reset_index(drop=True)
print(f"\nRows kept: {len(data)} of {rows_before} "
      f"(EXCLUDE_TEST_REMARK_ROWS = {EXCLUDE_TEST_REMARK_ROWS})")

                               rows  median_delay_min
remark = TEST                    26               4.5
remark starts with REPROGRAMM   118              19.0
same flight number and STA        8              10.0
more than 120 min early           3            -132.0
All flights, median delay: 10.0

Rows kept: 10863 of 10863 (EXCLUDE_TEST_REMARK_ROWS = False)


### Assessment of Reprogrammed Flights

The dataset contains 118 records whose remarks begin with "REPROGRAMM". These flights require additional scrutiny because a schedule revision changes the interpretation of the delay variable.

Since delay is calculated as the difference between block-on time and STA, it is necessary to determine whether STA corresponds to the original published schedule or the revised operational schedule. If STA reflects the rescheduled arrival time, the current delay definition remains valid. Otherwise, the target variable may overstate the operational delay for these flights.

## 7. Target

In [8]:
data["delayed"] = (data["delay_min"] > DELAY_THRESHOLD_MIN).astype(int)

print(f"Delayed (> {DELAY_THRESHOLD_MIN} min): {data['delayed'].sum()} of {len(data)} "
      f"({data['delayed'].mean():.1%})")
print()
print(data["delay_min"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1).to_string())
print()
for t in (0, 15, 30, 60):
    print(f"  > {t:>2} min: {(data['delay_min'] > t).mean():.1%}")

Delayed (> 15 min): 4520 of 10863 (41.6%)

count    10863.0
mean        19.8
std         46.1
min       -137.0
5%         -28.0
25%         -7.0
50%         10.0
75%         33.0
95%        105.0
max        580.0

  >  0 min: 63.8%
  > 15 min: 41.6%
  > 30 min: 26.5%
  > 60 min: 12.0%


## 8. Chronological split

In [9]:
def assign_split(sta):
    if sta.month <= 10:
        return "train"
    return "validation" if sta.month == 11 else "test"

data["split"] = data["sta_datetime"].apply(assign_split)

split_table = (
    data.groupby("split")
        .agg(flights=("delayed", "size"),
             delayed=("delayed", "sum"),
             delayed_share=("delayed", "mean"),
             first_sta=("sta_datetime", "min"),
             last_sta=("sta_datetime", "max"))
        .loc[["train", "validation", "test"]]
)
split_table["delayed_share"] = split_table["delayed_share"].round(3)
print(split_table.to_string())

assert split_table.loc["train", "last_sta"] < split_table.loc["validation", "first_sta"]
assert split_table.loc["validation", "last_sta"] < split_table.loc["test", "first_sta"]

monthly = data.groupby(data["sta_datetime"].dt.month)["delayed"].agg(["size", "mean"]).round(3)
monthly.index.name = "month"
print()
print(monthly.T.to_string())

            flights  delayed  delayed_share           first_sta            last_sta
split                                                                              
train          8994     3677          0.409 2025-01-01 00:05:00 2025-10-31 22:45:00
validation      902      320          0.355 2025-11-01 00:05:00 2025-11-30 22:30:00
test            967      523          0.541 2025-12-01 00:05:00 2025-12-31 23:25:00

month       1        2        3        4        5        6        7        8        9        10       11       12
size   992.000  876.000  919.000  874.000  888.000  836.000  915.000  930.000  873.000  891.000  902.000  967.000
mean     0.359    0.365    0.356    0.471    0.511    0.471    0.444    0.394    0.402    0.327    0.355    0.541


## 9. Features

In [10]:
data["sta_hour"] = data["sta_datetime"].dt.hour
data["sta_dayofweek"] = data["sta_datetime"].dt.dayofweek      # Monday = 0
data["is_weekend"] = (data["sta_dayofweek"] >= 5).astype(int)

NUMERIC = ["sta_hour", "sta_dayofweek", "is_weekend"]
FEATURES = CATEGORICAL + NUMERIC

FORBIDDEN = (set(ROLES["excluded: known only at or after arrival"])
             | set(ROLES["target source (never a feature)"])
             | {"delay_min", "delayed", "split", "remarks", "block_on_datetime"})
assert not (set(FEATURES) & FORBIDDEN), f"Leakage: {set(FEATURES) & FORBIDDEN}"
assert set(FEATURES) <= set(data.columns)

feature_table = pd.DataFrame({
    "feature": FEATURES,
    "type": ["categorical"] * len(CATEGORICAL) + ["numeric"] * len(NUMERIC),
    "known": ["schedule"] * 4 + ["scheduled time"] * 3,
})
print(feature_table.to_string(index=False))
print("\nLeakage check passed:", len(FEATURES), "features, none from the excluded list")

      feature        type          known
      airline categorical       schedule
flight_number categorical       schedule
aircraft_type categorical       schedule
       origin categorical       schedule
     sta_hour     numeric scheduled time
sta_dayofweek     numeric scheduled time
   is_weekend     numeric scheduled time

Leakage check passed: 7 features, none from the excluded list


## 10. Training and validation sets

In [11]:
MIN_FREQ = 30   # categories with fewer training rows are pooled into one 'rare' level

train = data[data["split"] == "train"].copy()
valid = data[data["split"] == "validation"].copy()
X_train, y_train = train[FEATURES], train["delayed"]
X_valid, y_valid = valid[FEATURES], valid["delayed"]
print("Train:", X_train.shape, "| Validation:", X_valid.shape)

seen = {c: set(train[c]) for c in CATEGORICAL}
coverage = pd.DataFrame({
    c: {
        "categories_in_train": train[c].nunique(),
        f"categories_under_{MIN_FREQ}_rows": int((train[c].value_counts() < MIN_FREQ).sum()),
        "valid_rows_unseen_in_train_pct": round(100 * (~valid[c].isin(seen[c])).mean(), 2),
    } for c in CATEGORICAL
}).T
print()
print(coverage.to_string())

Train: (8994, 7) | Validation: (902, 7)

               categories_in_train  categories_under_30_rows  valid_rows_unseen_in_train_pct
airline                       33.0                       6.0                            0.00
flight_number                176.0                     100.0                            1.22
aircraft_type                 38.0                      12.0                            1.00
origin                        66.0                      22.0                            0.00


## 11. Evaluation helper and majority-class baseline

In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def evaluate(y_true, y_pred, name):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "f1_delayed": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision_delayed": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_delayed": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

majority = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Majority class learned from training:", int(majority.classes_[majority.predict(X_train[:1])[0]]))

results = [evaluate(y_valid, majority.predict(X_valid), "majority class")]
results_valid = pd.DataFrame(results).set_index("model").round(3)
print()
print(results_valid.to_string())

Majority class learned from training: 0

                accuracy  macro_f1  f1_delayed  precision_delayed  recall_delayed
model                                                                            
majority class     0.645     0.392         0.0                0.0             0.0


## 12. Pipelines

In [13]:
import warnings
from itertools import product

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

THRESHOLD = 0.5
LINEAR_CATEGORICAL = CATEGORICAL + ["sta_hour", "sta_dayofweek"]

def linear_pipeline(C, min_freq):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=min_freq),
         LINEAR_CATEGORICAL),
    ])
    model = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=SEED)
    return Pipeline([("prep", prep), ("model", model)])

def tree_pipeline(min_freq, **params):
    prep = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               min_frequency=min_freq), CATEGORICAL),
        ("num", "passthrough", NUMERIC),
    ])
    model = HistGradientBoostingClassifier(
        categorical_features=list(range(len(CATEGORICAL))),
        class_weight="balanced", early_stopping=False, random_state=SEED, **params)
    return Pipeline([("prep", prep), ("model", model)])

def score_on_validation(pipe):
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_valid)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    row = evaluate(y_valid, pred, "")
    row["roc_auc"] = roc_auc_score(y_valid, proba)
    row["pr_auc"] = average_precision_score(y_valid, proba)
    return row, proba

for name, p in [("logistic", linear_pipeline(1.0, MIN_FREQ)),
                ("boosting", tree_pipeline(MIN_FREQ, max_iter=50))]:
    row, _ = score_on_validation(p)
    print(f"{name:9s} runs | macro F1 {row['macro_f1']:.3f} | ROC AUC {row['roc_auc']:.3f}")

logistic  runs | macro F1 0.638 | ROC AUC 0.698


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


boosting  runs | macro F1 0.633 | ROC AUC 0.698


## 13. Logistic regression tuning

In [14]:
rows = []
for C, mf in product([0.01, 0.03, 0.1, 0.3, 1.0], [10, 30, 60]):
    row, _ = score_on_validation(linear_pipeline(C, mf))
    rows.append({"C": C, "min_freq": mf, **{k: row[k] for k in ("macro_f1", "roc_auc", "pr_auc")}})

grid_lr = (pd.DataFrame(rows)
             .sort_values(["macro_f1", "roc_auc"], ascending=False)
             .reset_index(drop=True))
print(grid_lr.head(8).round(3).to_string())
best_lr = grid_lr.iloc[0]
print("\nSelected: C =", float(best_lr["C"]), "| min_freq =", int(best_lr["min_freq"]))

     C  min_freq  macro_f1  roc_auc  pr_auc
0  0.3        10     0.649    0.699   0.582
1  0.1        10     0.645    0.698   0.584
2  0.1        30     0.645    0.698   0.588
3  1.0        60     0.645    0.698   0.586
4  0.3        30     0.643    0.698   0.588
5  1.0        10     0.642    0.696   0.574
6  0.1        60     0.642    0.699   0.582
7  0.3        60     0.640    0.699   0.585

Selected: C = 0.3 | min_freq = 10


## 14. Gradient boosting tuning

In [15]:
rows = []
for lr, leaves, leaf_min, iters, mf in product([0.05, 0.1], [8, 16], [20, 60], [100, 300], [10, 30, 60]):
    params = dict(learning_rate=lr, max_leaf_nodes=leaves, min_samples_leaf=leaf_min, max_iter=iters)
    row, _ = score_on_validation(tree_pipeline(mf, **params))
    rows.append({**params, "min_freq": mf, **{k: row[k] for k in ("macro_f1", "roc_auc", "pr_auc")}})

grid_gb = (pd.DataFrame(rows)
             .sort_values(["macro_f1", "roc_auc"], ascending=False)
             .reset_index(drop=True))
print(grid_gb.head(8).round(3).to_string())
best_gb = grid_gb.iloc[0]
print("\nSelected: learning_rate =", float(best_gb["learning_rate"]),
      "| max_leaf_nodes =", int(best_gb["max_leaf_nodes"]),
      "| min_samples_leaf =", int(best_gb["min_samples_leaf"]),
      "| max_iter =", int(best_gb["max_iter"]),
      "| min_freq =", int(best_gb["min_freq"]))

   learning_rate  max_leaf_nodes  min_samples_leaf  max_iter  min_freq  macro_f1  roc_auc  pr_auc
0           0.05               8                60       100        60     0.651    0.700   0.586
1           0.05               8                20       100        60     0.646    0.699   0.584
2           0.10               8                20       100        60     0.644    0.700   0.585
3           0.10               8                60       100        60     0.643    0.701   0.591
4           0.10              16                60       100        10     0.643    0.698   0.580
5           0.05               8                60       300        60     0.642    0.697   0.588
6           0.05              16                60       100        60     0.642    0.703   0.587
7           0.10               8                20       100        10     0.641    0.694   0.574

Selected: learning_rate = 0.05 | max_leaf_nodes = 8 | min_samples_leaf = 60 | max_iter = 100 | min_freq = 60


## 15. Validation comparison

In [16]:
lr_pipe = linear_pipeline(best_lr["C"], int(best_lr["min_freq"]))
gb_pipe = tree_pipeline(
    int(best_gb["min_freq"]),
    learning_rate=best_gb["learning_rate"], max_leaf_nodes=int(best_gb["max_leaf_nodes"]),
    min_samples_leaf=int(best_gb["min_samples_leaf"]), max_iter=int(best_gb["max_iter"]))

table = []
matrices = {}
for name, pipe in [("logistic regression", lr_pipe), ("gradient boosting", gb_pipe)]:
    row, proba = score_on_validation(pipe)
    row["model"] = name
    table.append(row)
    matrices[name] = confusion_matrix(y_valid, (proba >= THRESHOLD).astype(int))

base = evaluate(y_valid, majority.predict(X_valid), "majority class")
base["roc_auc"], base["pr_auc"] = 0.5, y_valid.mean()      # no-skill reference values
results_valid = pd.DataFrame([base] + table).set_index("model").round(3)
print(results_valid.to_string())

for name, cm in matrices.items():
    print(f"\n{name}: rows = true (on time, delayed), columns = predicted")
    print(pd.DataFrame(cm, index=["on time", "delayed"], columns=["on time", "delayed"]).to_string())

                     accuracy  macro_f1  f1_delayed  precision_delayed  recall_delayed  roc_auc  pr_auc
model                                                                                                  
majority class          0.645     0.392       0.000              0.000           0.000    0.500   0.355
logistic regression     0.667     0.649       0.570              0.526           0.622    0.699   0.582
gradient boosting       0.666     0.651       0.578              0.524           0.644    0.700   0.586

logistic regression: rows = true (on time, delayed), columns = predicted
         on time  delayed
on time      403      179
delayed      121      199

gradient boosting: rows = true (on time, delayed), columns = predicted
         on time  delayed
on time      395      187
delayed      114      206


## 16. Final fit and single test evaluation

In [19]:
from pathlib import Path
from sklearn.base import clone

test = data[data["split"] == "test"].copy()
fit = data[data["split"].isin(["train", "validation"])].copy()
assert fit["sta_datetime"].max() < test["sta_datetime"].min(), "Test period overlaps the fitting period"

X_fit, y_fit = fit[FEATURES], fit["delayed"]
X_test, y_test = test[FEATURES], test["delayed"]
print("Final fit (Jan-Nov):", X_fit.shape, "| Test (Dec):", X_test.shape,
      "| delayed share in test:", round(y_test.mean(), 3))

final_models = {
    "majority class": DummyClassifier(strategy="most_frequent"),
    "logistic regression": clone(lr_pipe),      # same settings as selected on November
    "gradient boosting": clone(gb_pipe),        # same settings as selected on November
}

test_rows, test_scores = [], {}
for name, model in final_models.items():
    model.fit(X_fit, y_fit)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    row = evaluate(y_test, pred, name)
    row["roc_auc"] = roc_auc_score(y_test, proba)
    row["pr_auc"] = average_precision_score(y_test, proba)
    test_rows.append(row)
    test_scores[name] = (proba, pred)

results_test = pd.DataFrame(test_rows).set_index("model")
print(results_test.round(3).to_string())
print(f"\nNo-skill precision-recall AUC equals the delayed share: {y_test.mean():.3f}")

Path("results").mkdir(exist_ok=True)
results_test.round(4).to_csv("results/test_metrics.csv")
pd.DataFrame({
    "aibd_event_id": test["aibd_event_id"].to_numpy(),
    "y_true": y_test.to_numpy(),
    "proba_logistic": test_scores["logistic regression"][0],
    "proba_boosting": test_scores["gradient boosting"][0],
}).to_csv("../data/test_predictions.csv", index=False)      # row-level file, stays inside data/ (not published)

Final fit (Jan-Nov): (9896, 7) | Test (Dec): (967, 7) | delayed share in test: 0.541
                     accuracy  macro_f1  f1_delayed  precision_delayed  recall_delayed  roc_auc  pr_auc
model                                                                                                  
majority class          0.459     0.315       0.000              0.000           0.000    0.500   0.541
logistic regression     0.668     0.668       0.662              0.737           0.600    0.749   0.784
gradient boosting       0.644     0.644       0.657              0.686           0.631    0.739   0.773

No-skill precision-recall AUC equals the delayed share: 0.541


## 17. Uncertainty

In [ ]:
from sklearn.metrics import f1_score

rng = np.random.default_rng(SEED)
y_arr = y_test.to_numpy()
day = test["sta_datetime"].dt.date.to_numpy()
days = list(np.unique(day))
day_rows = {d: np.flatnonzero(day == d) for d in days}

names = ["logistic regression", "gradient boosting"]
B = 1000
store = {n: {"macro_f1": [], "roc_auc": []} for n in names}
for _ in range(B):
    pick = rng.integers(0, len(days), size=len(days))
    idx = np.concatenate([day_rows[days[i]] for i in pick])
    for n in names:
        proba, pred = test_scores[n]
        store[n]["macro_f1"].append(f1_score(y_arr[idx], pred[idx], average="macro"))
        store[n]["roc_auc"].append(roc_auc_score(y_arr[idx], proba[idx]))

rows = []
for n in names:
    for metric in ("macro_f1", "roc_auc"):
        lo, hi = np.percentile(store[n][metric], [2.5, 97.5])
        rows.append({"model": n, "metric": metric,
                     "estimate": results_test.loc[n, metric], "ci_low": lo, "ci_high": hi})

for metric in ("macro_f1", "roc_auc"):
    d = np.array(store["gradient boosting"][metric]) - np.array(store["logistic regression"][metric])
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"model": "boosting minus logistic", "metric": metric,
                 "estimate": results_test.loc["gradient boosting", metric]
                             - results_test.loc["logistic regression", metric],
                 "ci_low": lo, "ci_high": hi})
ci_table = pd.DataFrame(rows).round(3)
print(f"95% intervals from {B} resamples of the {len(days)} test days")
print(ci_table.to_string(index=False))
ci_table.to_csv("results/test_metrics_ci.csv", index=False)

## 18. Figures

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

Path("figures").mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, n in zip(axes, names):
    cm = confusion_matrix(y_test, test_scores[n][1])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=13,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["on time", "delayed"])
    ax.set_yticks([0, 1], ["on time", "delayed"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(n)
fig.suptitle("December 2025 test set, threshold 0.5")
fig.tight_layout()
fig.savefig("figures/confusion_matrices_test.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4))
for n in names:
    p, r, _ = precision_recall_curve(y_test, test_scores[n][0])
    ax.plot(r, p, label=n)
ax.axhline(y_test.mean(), color="grey", linestyle="--", label="no skill (delayed share)")
ax.set_xlabel("Recall (delayed)")
ax.set_ylabel("Precision (delayed)")
ax.set_ylim(0, 1)
ax.set_title("Precision-recall curves, December 2025")
ax.legend()
fig.tight_layout()
fig.savefig("figures/precision_recall_test.png", dpi=200)
plt.show()

## 19. Error analysis by time of day and airline

In [ ]:
ANONYMIZE_AIRLINES = True     # airline names are replaced by rank labels in printed and saved tables
MIN_GROUP = 30                # airlines with fewer test flights are pooled into one row

ea = test[["airline", "sta_hour", "delayed"]].copy()
ea["pred_lr"] = test_scores["logistic regression"][1]
ea["pred_gb"] = test_scores["gradient boosting"][1]
ea["err_lr"] = (ea["pred_lr"] != ea["delayed"]).astype(int)
ea["err_gb"] = (ea["pred_gb"] != ea["delayed"]).astype(int)

def summarize(frame, by):
    g = frame.groupby(by, observed=True)
    return pd.DataFrame({
        "flights": g.size(),
        "delayed_pct": 100 * g["delayed"].mean(),
        "lr_predicted_delayed_pct": 100 * g["pred_lr"].mean(),
        "lr_error_pct": 100 * g["err_lr"].mean(),
        "gb_predicted_delayed_pct": 100 * g["pred_gb"].mean(),
        "gb_error_pct": 100 * g["err_gb"].mean(),
    }).round(1)

ea["time_block"] = pd.cut(ea["sta_hour"], bins=[-1, 5, 11, 17, 23],
                          labels=["00-05", "06-11", "12-17", "18-23"])
by_time = summarize(ea, "time_block")
print("By scheduled time of day")
print(by_time.to_string())

counts = ea["airline"].value_counts()
big = counts[counts >= MIN_GROUP].index
other_label = f"OTHER (<{MIN_GROUP} flights each)"
ea["airline_group"] = np.where(ea["airline"].isin(big), ea["airline"], other_label)
by_airline = summarize(ea, "airline_group").sort_values("flights", ascending=False)
if ANONYMIZE_AIRLINES:
    named = [n for n in by_airline.index if n != other_label]
    relabel = {n: f"Airline {i + 1}" for i, n in enumerate(named)}
    by_airline.index = [relabel.get(n, n) for n in by_airline.index]
print("\nBy airline (Airline 1 = most test flights)")
print(by_airline.to_string())

by_time.to_csv("results/error_by_time_of_day.csv")
by_airline.to_csv("results/error_by_airline.csv")

## 20. Threshold sensitivity

In [ ]:
rows = []
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    row = {"threshold": t}
    for n, tag in [("logistic regression", "lr"), ("gradient boosting", "gb")]:
        pred = (test_scores[n][0] >= t).astype(int)
        row[f"{tag}_macro_f1"] = f1_score(y_test, pred, average="macro")
        row[f"{tag}_predicted_delayed_pct"] = 100 * pred.mean()
    rows.append(row)
threshold_table = pd.DataFrame(rows).round(3)
print(threshold_table.to_string(index=False))
print(f"\nActual delayed share in December: {100 * y_test.mean():.1f}%")
threshold_table.to_csv("results/threshold_sensitivity.csv", index=False)

## 21. Robustness to row exclusions

In [ ]:
remarks_norm = data["remarks"].astype("string")
is_test_remark = remarks_norm.eq("TEST").fillna(False)
is_rescheduled = remarks_norm.str.startswith("REPROGRAMM").fillna(False)

variants = {
    "all rows (reported above)": pd.Series(True, index=data.index),
    "without remark TEST": ~is_test_remark,
    "without TEST and rescheduled": ~(is_test_remark | is_rescheduled),
}

rows = []
for label, keep in variants.items():
    f_ = fit[keep.loc[fit.index].to_numpy()]
    t_ = test[keep.loc[test.index].to_numpy()]
    row = {"variant": label, "fit_rows": len(f_), "test_rows": len(t_)}
    for n, base in [("logistic regression", lr_pipe), ("gradient boosting", gb_pipe)]:
        m = clone(base).fit(f_[FEATURES], f_["delayed"])
        p = m.predict_proba(t_[FEATURES])[:, 1]
        tag = "lr" if n.startswith("logistic") else "gb"
        row[f"{tag}_macro_f1"] = f1_score(t_["delayed"], (p >= THRESHOLD).astype(int), average="macro")
        row[f"{tag}_roc_auc"] = roc_auc_score(t_["delayed"], p)
    rows.append(row)

robust = pd.DataFrame(rows).round(3)
print(robust.to_string(index=False))
same = np.isclose(robust.loc[0, "lr_macro_f1"], results_test.loc["logistic regression", "macro_f1"], atol=1e-3) \
   and np.isclose(robust.loc[0, "gb_macro_f1"], results_test.loc["gradient boosting", "macro_f1"], atol=1e-3)
print("\nFirst row reproduces Cell 16:", bool(same))
robust.to_csv("results/robustness_row_exclusions.csv", index=False)

In [ ]:
import sys, sklearn, matplotlib
info = (f"python {sys.version.split()[0]}\npandas {pd.__version__}\nnumpy {np.__version__}\n"
        f"scikit-learn {sklearn.__version__}\nmatplotlib {matplotlib.__version__}\nseed {SEED}\n")
print(info)
Path("results/run_info.txt").write_text(info)

In [20]:
from sklearn.metrics import precision_recall_curve
for n in ["logistic regression", "gradient boosting"]:
    p, r, _ = precision_recall_curve(y_test, test_scores[n][0])
    print(n, {f"recall>={k}": round(float(p[r >= k].max()), 3) for k in (0.5, 0.7, 0.9)})

logistic regression {'recall>=0.5': 0.773, 'recall>=0.7': 0.7, 'recall>=0.9': 0.645}
gradient boosting {'recall>=0.5': 0.751, 'recall>=0.7': 0.694, 'recall>=0.9': 0.639}
